# Caractérisation du jeu de données

Ce notebook produit les figures descriptives utilisées pour caractériser les données avant entraînement. Il ne contient aucune analyse de performance ni résultat expérimental.

## 1. Configuration

In [ ]:
from pathlib import Path
import json
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Chemins des fichiers
SCENARIOS_PATH = "data/scenarios_merged.jsonl"
SFT_TRAIN_PATH = "data/sft_train.jsonl"
GRPO_TRAIN_PATH = "data/grpo_train.jsonl"
VAL_PATH = "data/val.jsonl"
TEST_PATH = "data/test.jsonl"

# Répertoire de sortie des figures
FIGURES_DIR = "figures/"

# Palette de couleurs
COLOR_PRIMARY = "#1f4e79"
COLOR_SECONDARY = "#ed7d31"
COLOR_ACCENT = "#70ad47"
COLOR_WARNING = "#c00000"

FIGURES_PATH = Path(FIGURES_DIR)
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (8, 4.8),
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

## 2. Fonctions utilitaires

In [ ]:
CONFLICT_ORDER = ["IC", "CM", "IM", "IC+CM", "IC+IM", "CM+IM", "IC+CM+IM", "sans conflit"]
STRATEGY_ORDER = [f"S{i}" for i in range(1, 9)]

def read_jsonl(path):
    path = Path(path)
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def normalize_conflict_category(record):
    if record.get("conflict_category"):
        return str(record["conflict_category"]).replace("CM+IC", "IC+CM").replace("IC+IM+CM", "IC+CM+IM").replace("CM+IC+IM", "IC+CM+IM")
    conflicts = record.get("conflicts", []) or []
    conflict_types = sorted({str(conflict.get("type")) for conflict in conflicts if isinstance(conflict, dict) and conflict.get("type")})
    return "+".join(conflict_types) if conflict_types else "sans conflit"

def extract_json_object(text):
    if not isinstance(text, str):
        return None
    cleaned = re.sub(r"```json\s*|```\s*", "", text).strip()
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group())
    except json.JSONDecodeError:
        return None
    return parsed if isinstance(parsed, dict) else None

def extract_strategy_from_messages(record):
    messages = record.get("messages") or []
    if not messages:
        return None
    assistant_content = messages[-1].get("content", "") if isinstance(messages[-1], dict) else ""
    parsed = extract_json_object(assistant_content)
    if parsed and parsed.get("strategy"):
        match = re.search(r"S[1-8]", str(parsed["strategy"]))
        return match.group(0) if match else None
    match = re.search(r"\b(S[1-8])\b", assistant_content)
    return match.group(1) if match else None

def strategy_list(record):
    if "valid_strategies" in record and isinstance(record["valid_strategies"], list):
        return [item for item in record["valid_strategies"] if item in STRATEGY_ORDER]
    strategy = extract_strategy_from_messages(record)
    return [strategy] if strategy else []

def scenario_id(record, index):
    return str(record.get("query_id") or record.get("query") or index)

def iter_source_elements(record):
    for conflict in record.get("conflicts", []) or []:
        if not isinstance(conflict, dict):
            continue
        for element in conflict.get("conflictual_element", []) or []:
            if isinstance(element, dict):
                yield element
    for element in record.get("non_conflictual_elements", []) or []:
        if isinstance(element, dict):
            yield element

def count_retrieved_documents(record):
    if isinstance(record.get("documents"), list):
        return len(record["documents"])
    seen = set()
    for element in iter_source_elements(record):
        if element.get("source_type") == "parametric" or element.get("source_corpus") == "parametric":
            continue
        key = str(element.get("document_id") or element.get("id") or element.get("title") or "")
        if key:
            seen.add(key)
    return len(seen)

def annotate_vertical_bars(ax):
    for patch in ax.patches:
        height = patch.get_height()
        ax.annotate(f"{int(height)}", (patch.get_x() + patch.get_width() / 2, height), ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")

def annotate_horizontal_bars(ax):
    for patch in ax.patches:
        width = patch.get_width()
        ax.annotate(f"{int(width)}", (width, patch.get_y() + patch.get_height() / 2), ha="left", va="center", xytext=(4, 0), textcoords="offset points")

def save_current_figure(filename):
    path = FIGURES_PATH / filename
    plt.savefig(path, bbox_inches="tight", dpi=300)
    print(f"Figure sauvegardée : {path}")
    return path

## 3. Chargement des données

In [ ]:
scenarios = read_jsonl(SCENARIOS_PATH)
sft_train = read_jsonl(SFT_TRAIN_PATH)
grpo_train = read_jsonl(GRPO_TRAIN_PATH)
val_set = read_jsonl(VAL_PATH)
test_set = read_jsonl(TEST_PATH)

print(f"Scénarios complets : {len(scenarios)}")
print(f"SFT train : {len(sft_train)}")
print(f"GRPO train : {len(grpo_train)}")
print(f"Validation : {len(val_set)}")
print(f"Test : {len(test_set)}")

## 4. Figure 1 — Répartition des types de conflits

In [ ]:
conflict_counts = Counter(normalize_conflict_category(record) for record in scenarios)
conflict_df = pd.DataFrame({"type_conflit": list(conflict_counts.keys()), "n": list(conflict_counts.values())})
conflict_df = conflict_df.sort_values("n", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(conflict_df["type_conflit"], conflict_df["n"], color=COLOR_PRIMARY)
annotate_vertical_bars(ax)
ax.set_title("Répartition des types de conflits")
ax.set_xlabel("Type de conflit")
ax.set_ylabel("Nombre de scénarios")
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig_conflict_distribution = save_current_figure("fig_conflict_distribution.pdf")
plt.show()

## 5. Figure 2 — Répartition des stratégies dans le SFT

In [ ]:
strategy_counts = Counter()
for record in sft_train:
    for strategy in strategy_list(record):
        strategy_counts[strategy] += 1

strategy_values = [strategy_counts.get(strategy, 0) for strategy in STRATEGY_ORDER]
colors = plt.cm.Blues(np.linspace(0.45, 0.9, len(STRATEGY_ORDER)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(STRATEGY_ORDER, strategy_values, color=colors)
annotate_horizontal_bars(ax)
ax.set_title("Répartition des stratégies dans le SFT")
ax.set_xlabel("Nombre d'exemples")
ax.set_ylabel("Stratégie")
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)
ax.set_xlim(0, max(strategy_values + [1]) * 1.12)
plt.tight_layout()
fig_sft_strategy_distribution = save_current_figure("fig_sft_strategy_distribution.pdf")
plt.show()

## 6. Figure 3 — Nombre de stratégies valides par question

In [ ]:
strategies_by_question = defaultdict(set)
for index, record in enumerate(sft_train):
    qid = scenario_id(record, index)
    strategies_by_question[qid].update(strategy_list(record))

valid_strategy_counts = Counter(len(strategies) for strategies in strategies_by_question.values())
x_values = list(range(0, 9))
y_values = [valid_strategy_counts.get(value, 0) for value in x_values]
bar_colors = [COLOR_WARNING if value == 0 else "#bfbfbf" for value in x_values]

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.bar(x_values, y_values, color=bar_colors, edgecolor="white")
annotate_vertical_bars(ax)
ax.set_title("Nombre de stratégies valides par question")
ax.set_xlabel("Nombre de stratégies valides")
ax.set_ylabel("Nombre de questions")
ax.set_xticks(x_values)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
fig_valid_strategies_per_question = save_current_figure("fig_valid_strategies_per_question.pdf")
plt.show()

## 7. Figure 4 — Distribution des splits

In [ ]:
split_records = []
for split_name, records in [("Train", sft_train + grpo_train), ("Validation", val_set), ("Test", test_set)]:
    for record in records:
        split_records.append({"split": split_name, "type_conflit": normalize_conflict_category(record)})

split_df = pd.DataFrame(split_records)
split_table = pd.crosstab(split_df["type_conflit"], split_df["split"])
for column in ["Train", "Validation", "Test"]:
    if column not in split_table.columns:
        split_table[column] = 0
split_table = split_table[["Train", "Validation", "Test"]]
split_table = split_table.loc[split_table.sum(axis=1).sort_values(ascending=False).index]
split_percent = split_table.div(split_table.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(split_percent))
for split_name, color in [("Train", COLOR_PRIMARY), ("Validation", COLOR_SECONDARY), ("Test", COLOR_ACCENT)]:
    values = split_percent[split_name].to_numpy()
    ax.bar(split_percent.index, values, bottom=bottom, label=split_name, color=color)
    bottom += values

ax.set_title("Distribution des splits par type de conflit")
ax.set_xlabel("Type de conflit")
ax.set_ylabel("Pourcentage")
ax.set_ylim(0, 100)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=3, frameon=False)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig_split_distribution = save_current_figure("fig_split_distribution.pdf")
plt.show()

## 8. Figure 5 — Nombre de documents par scénario

In [ ]:
documents_per_scenario = [count_retrieved_documents(record) for record in scenarios]
median_documents = float(np.median(documents_per_scenario)) if documents_per_scenario else 0.0
bins = np.arange(0, max(documents_per_scenario + [0]) + 2) - 0.5

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.hist(documents_per_scenario, bins=bins, color="#9dc3e6", edgecolor="white")
ax.axvline(median_documents, color=COLOR_WARNING, linestyle="--", linewidth=1.6, label=f"Médiane = {median_documents:.1f}")
ax.set_title("Nombre de documents récupérés par scénario")
ax.set_xlabel("Nombre de documents")
ax.set_ylabel("Nombre de scénarios")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
fig_documents_per_scenario = save_current_figure("fig_documents_per_scenario.pdf")
plt.show()

## 9. Récapitulatif des fichiers générés

In [ ]:
generated_figures = [
    fig_conflict_distribution,
    fig_sft_strategy_distribution,
    fig_valid_strategies_per_question,
    fig_split_distribution,
    fig_documents_per_scenario,
]

print("Figures générées :")
for path in generated_figures:
    print(f"- {path}")